In [28]:
from seedler import Planter, Sprout, Fire, Nursery

ImportError: cannot import name 'Planter' from 'seedler' (/Users/dallinwolfer/Documents/Code/DS495R/seedler-repo/seedler/__init__.py)

In [3]:
suits = ["H", 'S', 'D', 'C']
values = ["A", "K", "Q", "J", '10', '9', '8', '7', '6', '5', '4', '3', '2']

base_deck = [f"{val}{suit}" for suit in suits for val in values]

print(base_deck)

['AH', 'KH', 'QH', 'JH', '10H', '9H', '8H', '7H', '6H', '5H', '4H', '3H', '2H', 'AS', 'KS', 'QS', 'JS', '10S', '9S', '8S', '7S', '6S', '5S', '4S', '3S', '2S', 'AD', 'KD', 'QD', 'JD', '10D', '9D', '8D', '7D', '6D', '5D', '4D', '3D', '2D', 'AC', 'KC', 'QC', 'JC', '10C', '9C', '8C', '7C', '6C', '5C', '4C', '3C', '2C']


In [4]:
ROYAL_FLUSH = 0
STRAIGHT_FLUSH = 1
FOUR_KIND = 2
FULL_HOUSE = 3
FLUSH = 4
STRAIGHT = 5
THREE_KIND = 6
TWO_PAIR = 7
TWO_KIND = 8
HIGH_CARD = 9

In [5]:
from collections import Counter

class PokerLab(Planter):
    # Convert card string to numeric value
    def __card_value(self, card):
        val = card[:-1]
        if val == "A":
            return 14  # Ace high
        if val == "K":
            return 13
        if val == "Q":
            return 12
        if val == "J":
            return 11
        return int(val)

    # Check if all cards have same suit
    def __flush(self, hand):
        suits = {c[-1] for c in hand}
        return len(suits) == 1

    # Check if hand is straight
    def __straight(self, hand):
        vals = sorted(self.__card_value(c) for c in hand)
        # Special case for Ace-low straight
        if vals[-1] == 14:
            vals_ace_low = [1 if v == 14 else v for v in vals]
            if max(vals_ace_low) - min(vals_ace_low) == len(vals)-1 and len(set(vals_ace_low)) == len(vals):
                return True
        return max(vals) - min(vals) == len(vals)-1 and len(set(vals)) == len(vals)

    # Get counts of card values
    def __kind_counts(self, hand):
        vals = [c[:-1] for c in hand]
        return Counter(vals)

    def plant(self, sprout: Sprout):
        deck = base_deck.copy()
        hand = [deck.pop(sprout.growth(0, len(deck)-1)) for _ in range(5)]

        flush = self.__flush(hand)
        straight = self.__straight(hand)
        counts = self.__kind_counts(hand)
        most_common = counts.most_common()

        vals = set(counts.keys())

        # Royal flush
        if flush and straight and {"A","K","Q","J","10"}.issubset(vals):
            sprout.add_bud(ROYAL_FLUSH)
            return

        if flush and straight:
            sprout.add_bud(STRAIGHT_FLUSH)
            return

        if most_common[0][1] == 4:
            sprout.add_bud(FOUR_KIND)
            return

        if most_common[0][1] == 3 and most_common[1][1] == 2:
            sprout.add_bud(FULL_HOUSE)
            return

        if flush:
            sprout.add_bud(FLUSH)
            return

        if straight:
            sprout.add_bud(STRAIGHT)
            return

        if most_common[0][1] == 3:
            sprout.add_bud(THREE_KIND)
            return

        if most_common[0][1] == 2:
            if most_common[1][1] == 2:
                sprout.add_bud(TWO_PAIR)
            else:
                sprout.add_bud(TWO_KIND)
            return

        sprout.add_bud(HIGH_CARD)

In [6]:
class PokerFire(Fire):
    def purge(self, sprout: Sprout):
        return sprout.get_bud_count(HIGH_CARD) != 0

In [7]:
lab = PokerLab()
planter = lab.find_seeds(fire=PokerFire(), minimum=0, maximum=50_000)

In [8]:
df = Nursery.pandas(planter, expanded=True)
display(df)

,Seed,8,7,6,5,3,4,2,1
0,0,1,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0
2,5,1,0,0,0,0,0,0,0
3,7,1,0,0,0,0,0,0,0
4,10,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
25157,49991,1,0,0,0,0,0,0,0
25158,49992,1,0,0,0,0,0,0,0
25159,49994,1,0,0,0,0,0,0,0
25160,49995,1,0,0,0,0,0,0,0


In [9]:
df.rename(columns={
    ROYAL_FLUSH: "RoyalFlush",
    STRAIGHT_FLUSH: "StraightFlush",
    FOUR_KIND: "FourKind",
    FULL_HOUSE: "FullHouse",
    FLUSH: "Flush",
    STRAIGHT: "Straight",
    THREE_KIND: "ThreeKind",
    TWO_PAIR: "TwoPair",
    TWO_KIND: "TwoKind",
    HIGH_CARD: "HighCard",
}, inplace=True)

df

,Seed,TwoKind,TwoPair,ThreeKind,Straight,FullHouse,Flush,FourKind,StraightFlush
0,0,1,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0
2,5,1,0,0,0,0,0,0,0
3,7,1,0,0,0,0,0,0,0
4,10,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
25157,49991,1,0,0,0,0,0,0,0
25158,49992,1,0,0,0,0,0,0,0
25159,49994,1,0,0,0,0,0,0,0
25160,49995,1,0,0,0,0,0,0,0


# Graphing

In [12]:
occurrences = (
    df
    .drop('Seed', axis=1)
    .sum().rename("count")
    .reset_index()
    .rename(columns={'index':'hand'})
    .sort_values(by='count', ascending=False)
)
occurrences

,hand,count
0,TwoKind,21288
1,TwoPair,2368
2,ThreeKind,1080
3,Straight,217
5,Flush,102
4,FullHouse,91
6,FourKind,15
7,StraightFlush,1


In [14]:
from lets_plot import *
LetsPlot.setup_html()

/Users/dallinwolfer/Documents/Code/DS495R/seedler-repo/.venv/lib/python3.14/site-packages/lets_plot/plot/annotation.py:592: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  .line(r'\(R\^2=\)@..r2..')\\
/Users/dallinwolfer/Documents/Code/DS495R/seedler-repo/.venv/lib/python3.14/site-packages/lets_plot/plot/annotation.py:644: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  .line(r'\(R\^2=\)@..r2..')\\


In [15]:
(
    ggplot(
        occurrences,
        aes(x="hand", y='count'),
    )
    + geom_bar(stat='identity')
    + scale_y_log10()
)